In [1]:
import polars as pl
import pandas as pd

In [2]:
df = (
    pl.scan_csv("data/rating_complete.csv")
    .collect()
)

In [3]:
df.head()

user_id,anime_id,rating
i64,i64,i64
0,430,9
0,1004,5
0,3010,7
0,570,7
0,2762,9


In [4]:
df.shape

(57633278, 3)

In [5]:
df = pd.read_csv("data/rating_complete.csv")
df.shape

(57633278, 3)

In [6]:
import pandas as pd

from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse, mae

# Keep only required columns
ratings = df[['user_id', 'anime_id', 'rating']]

reader = Reader(rating_scale=(ratings.rating.min(), ratings.rating.max()))

data = Dataset.load_from_df(
    ratings[['user_id', 'anime_id', 'rating']],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model.fit(trainset)

predictions = model.test(testset)

print("RMSE:", rmse(predictions))
print("MAE :", mae(predictions))

RMSE: 1.1136
RMSE: 1.1135621604112111
MAE:  0.8212
MAE : 0.821232949743209


### BENCHMARK

In [7]:
from surprise.accuracy import rmse, mae

rmse_value = rmse(predictions, verbose=False)
mae_value = mae(predictions, verbose=False)

rating_range = ratings["rating"].max() - ratings["rating"].min()

rmse_pct = (rmse_value / rating_range) * 100
mae_pct = (mae_value / rating_range) * 100

print(f"RMSE: {rmse_value:.3f} ({rmse_pct:.2f}%)")
print(f"MAE : {mae_value:.3f} ({mae_pct:.2f}%)")

RMSE: 1.114 (12.37%)
MAE : 0.821 (9.12%)
